In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
print("hi")

In [ ]:
!pip install tensorflow keras numpy matplotlib scikit-image opencv-python gdown scikit-learn

In [ ]:
import gdown
import zipfile
import os

# Download from Google Drive
file_id = "1uJmDZw649XS-r-dYs9WD-OPwF_TIroVw"
output = "dataset.zip"
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

# Extract
with zipfile.ZipFile("dataset.zip", "r") as z:
    z.extractall("dataset")

# Check structure
for root, dirs, files in os.walk("dataset"):
    level = root.replace("dataset", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:5]:
            print(f"{indent}  {f}")

In [ ]:
import cv2
import numpy as np
import os

def load_npy_images(directory, target_size=None):
    images = []
    files = sorted([f for f in os.listdir(directory) if f.endswith('.npy')])
    
    for fname in files:
        fpath = os.path.join(directory, fname)
        img = np.load(fpath)  # shape: (1, H, W)
        
        # Fix channel-first -> channel-last
        if img.ndim == 3 and img.shape[0] == 1:
            img = img[0]  # (H, W)
        
        # Resize if needed
        if target_size:
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_CUBIC)
        
        img = img[:, :, np.newaxis]  # → (H, W, 1)
        images.append(img.astype(np.float32))
    
    return np.array(images)


HR_DIR = "/kaggle/working/dataset/Dataset/HR"
LR_DIR = "/kaggle/working/dataset/Dataset/LR"

HR_SIZE = (150, 150)  # (W, H) for cv2.resize
LR_SIZE = (75, 75)

print("Loading HR images...")
hr_images = load_npy_images(HR_DIR, target_size=HR_SIZE)

print("Loading LR images...")
lr_images = load_npy_images(LR_DIR, target_size=LR_SIZE)

print(f"\n HR images: {hr_images.shape}")   # should be (10000, 150, 150, 1)
print(f" LR images: {lr_images.shape}")    # should be (10000, 75, 75, 1)
print(f"HR min/max: {hr_images.min():.4f} / {hr_images.max():.4f}")
print(f"LR min/max: {lr_images.min():.4f} / {lr_images.max():.4f}")


In [ ]:
import numpy as np
import cv2

def normalize(images):
    """Normalize to [0, 1] per dataset"""
    vmin = images.min()
    vmax = images.max()
    return (images - vmin) / (vmax - vmin + 1e-8)

def upscale_images(images, target_size=(150, 150)):
    """Bicubic upscale LR 75x75 → 150x150"""
    out = []
    for img in images:
        up = cv2.resize(img.squeeze(), target_size, interpolation=cv2.INTER_CUBIC)
        out.append(up[:, :, np.newaxis])
    return np.array(out, dtype=np.float32)

# Normalizing both (HR already 0-1, LR has slight out-of-range values)
hr_images = normalize(hr_images)
lr_images = normalize(lr_images)

# Upscaling LR 75x75 -> 150x150 (model input = upscaled LR, target = HR)
print("Upscaling LR images 75->150...")
lr_upscaled = upscale_images(lr_images, target_size=(150, 150))

print(f"\n LR upscaled : {lr_upscaled.shape}")   # (10000, 150, 150, 1)
print(f" HR target   : {hr_images.shape}")        # (10000, 150, 150, 1)
print(f"LR upscaled min/max: {lr_upscaled.min():.4f} / {lr_upscaled.max():.4f}")
print(f"HR min/max         : {hr_images.min():.4f} / {hr_images.max():.4f}")

# Match counts
n = min(len(lr_upscaled), len(hr_images))
lr_upscaled = lr_upscaled[:n]
hr_images   = hr_images[:n]
print(f"\n Matched pairs: {n}")

In [ ]:
from sklearn.model_selection import train_test_split

# 90:10 spliting as per submission guidelines
X_train, X_test, y_train, y_test = train_test_split(
    lr_upscaled, hr_images, test_size=0.1, random_state=42
)

# Further spliting test into val+test (50:50 of the 10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42
)

print(f"Train : {X_train.shape}")   # 9000
print(f"Val   : {X_val.shape}")     # 500
print(f"Test  : {X_test.shape}")    # 500

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

def residual_channel_attention_block(x, filters=32, reduction=8):
    shortcut = x
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    gap = layers.GlobalAveragePooling2D()(x)
    gap = layers.Reshape((1, 1, filters))(gap)
    gap = layers.Conv2D(filters // reduction, 1, activation='relu')(gap)
    gap = layers.Conv2D(filters, 1, activation='sigmoid')(gap)
    x = layers.Multiply()([x, gap])
    x = layers.Add()([x, shortcut])
    return x

def build_pretrained_encoder(input_shape):
    inp = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 9, padding='same', activation='relu')(inp)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    encoder = Model(inp, x, name="pretrained_encoder")
    return encoder

def build_sr_model(input_shape=(150, 150, 1), n_residual_blocks=8, filters=32):
    inp = keras.Input(shape=input_shape, name="lr_input")

    encoder = build_pretrained_encoder(input_shape)
    encoder.trainable = False
    feat = encoder(inp)

    x = feat
    for _ in range(n_residual_blocks):
        x = residual_channel_attention_block(x, filters=filters)

    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.Add()([x, feat])

    skip = layers.Conv2D(filters, 1, padding='same')(inp)
    x = layers.Add()([x, skip])

    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(1, 3, padding='same', activation='sigmoid', name="sr_output")(x)

    model = Model(inp, x, name="RCAN_SR_Lite")
    return model, encoder

INPUT_SHAPE = (150, 150, 1)
sr_model, encoder = build_sr_model(INPUT_SHAPE, n_residual_blocks=8, filters=32)
sr_model.summary()

In [ ]:
import tensorflow as tf

def ssim_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def perceptual_mse_ssim_loss(y_true, y_pred):
    mse  = tf.reduce_mean(tf.square(y_true - y_pred))
    ssim = ssim_loss(y_true, y_pred)
    return 0.7 * mse + 0.3 * ssim

def psnr_metric(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim_metric(y_true, y_pred):
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

print(" Custom loss & metrics defined!")

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
import os
import tensorflow as tf

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

#  Using tf.data to avoid RAM overflow 
BATCH_SIZE = 8  # small batch for GPU memory

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))\
    .shuffle(500).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))\
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Phase 1: Encoder frozen
encoder.trainable = False

sr_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=perceptual_mse_ssim_loss,
    metrics=[psnr_metric, ssim_metric, 'mse']
)

callbacks_p1 = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint("/kaggle/working/checkpoints/sr_phase1_best.keras",
                    monitor='val_loss', save_best_only=True, verbose=1)
]


tf.keras.backend.clear_session()
import gc; gc.collect()

sr_model, encoder = build_sr_model(INPUT_SHAPE, n_residual_blocks=8, filters=32)
encoder.trainable = False
sr_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=perceptual_mse_ssim_loss,
    metrics=[psnr_metric, ssim_metric, 'mse']
)

print(" Phase 1: Encoder FROZEN. Training SR head only for now.")
history_p1 = sr_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks_p1,
    verbose=1
)

In [ ]:
import gc
gc.collect()
tf.keras.backend.clear_session()

# Reload best phase 1 model
sr_model = keras.models.load_model(
    "/kaggle/working/checkpoints/sr_phase1_best.keras",
    custom_objects={
        'perceptual_mse_ssim_loss': perceptual_mse_ssim_loss,
        'psnr_metric': psnr_metric,
        'ssim_metric': ssim_metric
    }
)

# Phase 2: Unfreeze ALL layers
for layer in sr_model.layers:
    layer.trainable = True

sr_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # 10x lower LR
    loss=perceptual_mse_ssim_loss,
    metrics=[psnr_metric, ssim_metric, 'mse']
)

callbacks_p2 = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-7, verbose=1),
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint("/kaggle/working/checkpoints/sr_phase2_best.keras",
                    monitor='val_loss', save_best_only=True, verbose=1)
]

print(" Phase 2: FULL fine-tuning (encoder + SR head) now yay")
history_p2 = sr_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks_p2,
    verbose=1
)

In [ ]:
import shutil

# Copy weights to output for easy download bcz i am lazy today
shutil.copy(
    "/kaggle/working/checkpoints/sr_phase2_best.keras",
    "/kaggle/working/sr_final_model.keras"
)
print(" Model ready to download from Output panel!")

In [ ]:
import shutil
import os

# Zip the model weights
shutil.make_archive(
    "/kaggle/working/model_weights",  # output zip name
    'zip',
    "/kaggle/working/checkpoints"     # folder to zip
)

print(" Done! Download 'model_weights.zip' from Output panel")

In [ ]:
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
import numpy as np

# Loads best model
sr_model = keras.models.load_model(
    "/kaggle/working/checkpoints/sr_phase2_best.keras",
    custom_objects={
        'perceptual_mse_ssim_loss': perceptual_mse_ssim_loss,
        'psnr_metric': psnr_metric,
        'ssim_metric': ssim_metric
    }
)

# Predict on full test set (500 images)
y_pred = sr_model.predict(X_test, batch_size=8, verbose=1)

mse_scores, ssim_scores, psnr_scores = [], [], []

for gt, pred in zip(y_test, y_pred):
    gt_   = gt.squeeze()
    pred_ = np.clip(pred.squeeze(), 0, 1)
    mse_scores.append(np.mean((gt_ - pred_) ** 2))
    ssim_scores.append(compare_ssim(gt_, pred_, data_range=1.0))
    psnr_scores.append(compare_psnr(gt_, pred_, data_range=1.0))

print(f"  MSE  : {np.mean(mse_scores):.6f} ± {np.std(mse_scores):.6f}")
print(f"  SSIM : {np.mean(ssim_scores):.4f}  ± {np.std(ssim_scores):.4f}")
print(f"  PSNR : {np.mean(psnr_scores):.2f} dB ± {np.std(psnr_scores):.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_show = 5
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))

indices = np.random.choice(len(X_test), n_show, replace=False)

for i, idx in enumerate(indices):
    lr_img = X_test[idx].squeeze()
    sr_img = np.clip(y_pred[idx].squeeze(), 0, 1)
    hr_img = y_test[idx].squeeze()

    psnr_val = compare_psnr(hr_img, sr_img, data_range=1.0)
    ssim_val = compare_ssim(hr_img, sr_img, data_range=1.0)

    axes[i, 0].imshow(lr_img, cmap='inferno')
    axes[i, 0].set_title('LR Input (Bicubic)')
    axes[i, 1].imshow(sr_img, cmap='inferno')
    axes[i, 1].set_title(f'SR Output\nPSNR={psnr_val:.1f}dB SSIM={ssim_val:.3f}')
    axes[i, 2].imshow(hr_img, cmap='inferno')
    axes[i, 2].set_title('HR Ground Truth')

    for ax in axes[i]:
        ax.axis('off')

plt.suptitle('Super-Resolution — Strong Lensing Images', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/visual_results.png', dpi=150)
plt.show()
print(" Visual saved yayy")

In [ ]:
import shutil

# Saving model
sr_model.save("/kaggle/working/sr_strong_lensing_final.keras")
print(" Model saved")

# Ziping everything for me to download
shutil.make_archive("/kaggle/working/all_outputs", 'zip', "/kaggle/working/checkpoints")
print(" Weights zipped")

print("\n Files ready to download:")
print("  - sr_strong_lensing_final.keras")
print("  - visual_results.png")
print("  - all_outputs.zip (weights)")